# Ridge, Lasso and ElasticNet from scratch 
Covers: Ridge (closed form + gradient descent), Lasso (subgradient descent as in the doc,
and coordinate descent / ISTA which give exact zeros), ElasticNet, standardisation,
choosing alpha by validation/CV, coefficient paths, sparsity recovery, correlated features.
Loss (doc): sum((y - Xw)^2) + alpha * penalty. Below we use the 1/m-scaled version
J = (1/m)||y - Xw - b||^2 + alpha * penalty, matching the doc's gradient code.

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(3)

## 1. Data: 20 features, only 5 informative; few samples -> overfitting for plain OLS

In [ ]:
def make_data(n, p=20, k=5, noise=2.0, seed=0):
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, p))
    w = np.zeros(p); w[:k] = [5, -4, 3, -2, 1.5]
    return X, X @ w + 3 + r.normal(0, noise, n), w

X_all, y_all, w_true = make_data(120)
X_tr, y_tr = X_all[:40], y_all[:40]
X_val, y_val = X_all[40:80], y_all[40:80]
X_te, y_te = X_all[80:], y_all[80:]

def standardize(X, mu=None, sd=None):
    if mu is None:
        mu, sd = X.mean(0), X.std(0)
    return (X - mu) / sd, mu, sd

Xs_tr, mu, sd = standardize(X_tr)
Xs_val, Xs_te = (X_val - mu) / sd, (X_te - mu) / sd
ymean = y_tr.mean()          # centring y == unpenalised intercept
def mse(y, p): return np.mean((y - p) ** 2)
def r2(y, p): return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

## 2. Ridge: closed form  w = (X^T X + m*alpha*I)^-1 X^T y   and gradient descent
GD gradient (doc): (2/m) X^T err + 2*alpha*w.

In [ ]:
class Ridge:
    def __init__(self, alpha=1.0, solver='closed', lr=0.05, n_iters=2000):
        self.alpha, self.solver, self.lr, self.n_iters = alpha, solver, lr, n_iters
    def fit(self, X, y):
        m, k = X.shape; self.b0 = y.mean(); yc = y - self.b0   # X assumed standardised
        if self.solver == 'closed':
            self.w = np.linalg.solve(X.T @ X + m * self.alpha * np.eye(k), X.T @ yc)
        else:
            self.w = np.zeros(k)
            for _ in range(self.n_iters):
                err = X @ self.w - yc
                self.w -= self.lr * ((2 / m) * X.T @ err + 2 * self.alpha * self.w)
        return self
    def predict(self, X): return X @ self.w + self.b0

r_cf = Ridge(0.1).fit(Xs_tr, y_tr); r_gd = Ridge(0.1, solver='gd').fit(Xs_tr, y_tr)
print("Ridge closed vs GD max |diff|:", np.abs(r_cf.w - r_gd.w).max().round(6))
print("alpha=0 -> OLS test MSE:", mse(y_te, Ridge(0.0).fit(Xs_tr, y_tr).predict(Xs_te)).round(2),
      "| Ridge(0.1) test MSE:", mse(y_te, r_cf.predict(Xs_te)).round(2))

## 3. Lasso
(a) Subgradient descent as written in the doc: gradient = (2/m) X^T err + alpha*sign(w). Weights
    oscillate near zero but rarely become exactly 0.
(b) Coordinate descent with soft-thresholding: exact zeros.
(c) ISTA (proximal gradient): w <- S(w - lr*grad_smooth, lr*alpha).

In [ ]:
def soft_threshold(z, t):
    """S(z,t) = sign(z) * max(|z|-t, 0): the proximal operator of the L1 norm."""
    return np.sign(z) * np.maximum(np.abs(z) - t, 0.0)

class ElasticNet:
    """
    Minimises (1/m)||y-Xw||^2 + alpha*(l1_ratio*||w||_1 + (1-l1_ratio)*||w||_2^2)
    l1_ratio=1 -> Lasso, l1_ratio=0 -> Ridge. solver: 'cd' (coordinate descent),
    'ista' (proximal GD) or 'subgrad' (doc-style, no exact zeros).
    """
    def __init__(self, alpha=0.1, l1_ratio=0.5, solver='cd', n_iters=500, lr=0.05):
        self.alpha, self.l1_ratio, self.solver, self.n_iters, self.lr = alpha, l1_ratio, solver, n_iters, lr

    def cost(self, X, yc):
        w = self.w; m = len(yc)
        return np.mean((yc - X @ w) ** 2) + self.alpha * (self.l1_ratio * np.abs(w).sum() + (1 - self.l1_ratio) * w @ w)

    def fit(self, X, y):
        m, k = X.shape; self.b0 = y.mean(); yc = y - self.b0
        a1, a2 = self.alpha * self.l1_ratio, self.alpha * (1 - self.l1_ratio)
        self.w = np.zeros(k); self.history = []
        if self.solver == 'cd':
            # per-coordinate exact minimiser: w_j = S(2/m x_j.r, a1) / (2/m x_j.x_j + 2 a2)
            for _ in range(self.n_iters):
                for j in range(k):
                    r = yc - X @ self.w + X[:, j] * self.w[j]
                    num = soft_threshold(2 / m * X[:, j] @ r, a1)
                    self.w[j] = num / (2 / m * X[:, j] @ X[:, j] + 2 * a2)
                self.history.append(self.cost(X, yc))
        else:
            for _ in range(self.n_iters):
                grad = (2 / m) * X.T @ (X @ self.w - yc) + 2 * a2 * self.w
                if self.solver == 'ista':
                    self.w = soft_threshold(self.w - self.lr * grad, self.lr * a1)
                else:                                     # subgradient
                    self.w = self.w - self.lr * (grad + a1 * np.sign(self.w))
                self.history.append(self.cost(X, yc))
        return self

    def predict(self, X): return X @ self.w + self.b0

nnz = lambda w: int((np.abs(w) > 1e-8).sum())
for s in ['subgrad', 'ista', 'cd']:
    m_ = ElasticNet(alpha=0.5, l1_ratio=1.0, solver=s, n_iters=1000 if s != 'cd' else 200).fit(Xs_tr, y_tr)
    print(f"Lasso[{s:7s}] exact zeros: {20 - nnz(m_.w):2d}/20, smallest |w|: {np.abs(m_.w).min():.2e}, "
          f"final cost {m_.history[-1]:.3f}")
lasso = ElasticNet(alpha=0.5, l1_ratio=1.0).fit(Xs_tr, y_tr)
print("Lasso support:", np.nonzero(np.abs(lasso.w) > 1e-8)[0], "(true: 0-4)")

## 4. ElasticNet reduces to Ridge / Lasso at the extremes

In [ ]:
en0 = ElasticNet(alpha=0.1, l1_ratio=0.0, solver='cd', n_iters=300).fit(Xs_tr, y_tr)
# our Ridge closed-form uses the same objective scaled by 1/m*alpha -> compare through equivalent objective
w_ridge_equiv = np.linalg.solve(Xs_tr.T @ Xs_tr + 40 * 0.1 * np.eye(20), Xs_tr.T @ (y_tr - y_tr.mean()))
print("ElasticNet(l1_ratio=0) vs Ridge diff:", np.abs(en0.w - w_ridge_equiv).max().round(6))
en1 = ElasticNet(alpha=0.5, l1_ratio=1.0).fit(Xs_tr, y_tr)
print("ElasticNet(l1_ratio=1) vs Lasso diff:", np.abs(en1.w - lasso.w).max().round(8))

## 5. Coefficient paths vs alpha (Ridge shrinks smoothly, Lasso zeroes out, ElasticNet in between)

In [ ]:
alphas = np.logspace(-3, 1, 25)
paths = {
    'Ridge': np.array([Ridge(a).fit(Xs_tr, y_tr).w for a in alphas]),
    'Lasso': np.array([ElasticNet(a, 1.0, n_iters=100).fit(Xs_tr, y_tr).w for a in alphas]),
    'ElasticNet (0.5)': np.array([ElasticNet(a, 0.5, n_iters=100).fit(Xs_tr, y_tr).w for a in alphas]),
}
fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for a_, (nm, P) in zip(ax, paths.items()):
    for j in range(20):
        a_.semilogx(alphas, P[:, j], color='tab:red' if j < 5 else 'gray', lw=1.5 if j < 5 else .8)
    a_.set_title(nm); a_.set_xlabel('alpha')
ax[0].set_ylabel('coefficient (red = informative)')
plt.tight_layout(); plt.savefig('plots/01_coef_paths.png', dpi=120); plt.close()

## 6. Exercise 1: number of zero coefficients vs alpha for Lasso

In [ ]:
zeros = [20 - nnz(P) for P in paths['Lasso']]
plt.figure(figsize=(6, 4)); plt.semilogx(alphas, zeros, 'o-'); plt.xlabel('alpha'); plt.ylabel('# zero coefficients')
plt.title('Lasso sparsity'); plt.tight_layout(); plt.savefig('plots/02_lasso_sparsity.png', dpi=120); plt.close()

## 7. Choosing alpha: validation curve (U shape) and k-fold CV

In [ ]:
def val_curve(make, alphas):
    tr_e, va_e = [], []
    for a in alphas:
        m_ = make(a).fit(Xs_tr, y_tr)
        tr_e.append(mse(y_tr, m_.predict(Xs_tr))); va_e.append(mse(y_val, m_.predict(Xs_val)))
    return np.array(tr_e), np.array(va_e)

al = np.logspace(-4, 1, 30)
curves = {'Ridge': val_curve(lambda a: Ridge(a), al),
          'Lasso': val_curve(lambda a: ElasticNet(a, 1.0, n_iters=100), al),
          'ElasticNet': val_curve(lambda a: ElasticNet(a, 0.5, n_iters=100), al)}
fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for a_, (nm, (t, v)) in zip(ax, curves.items()):
    a_.semilogx(al, t, label='train'); a_.semilogx(al, v, label='validation')
    a_.axvline(al[np.argmin(v)], c='k', ls='--', lw=.7); a_.set_title(f'{nm}: best alpha={al[np.argmin(v)]:.3g}')
    a_.set_xlabel('alpha'); a_.legend()
ax[0].set_ylabel('MSE'); plt.tight_layout(); plt.savefig('plots/03_validation_curves.png', dpi=120); plt.close()

def kfold_cv(make, X, y, k=5):
    folds = np.array_split(np.arange(len(y)), k); errs = []
    for i in range(k):
        t = np.concatenate([folds[j] for j in range(k) if j != i]); v = folds[i]
        Xt, mu_, sd_ = standardize(X[t]); m_ = make().fit(Xt, y[t])
        errs.append(mse(y[v], m_.predict((X[v] - mu_) / sd_)))
    return np.mean(errs)

X_cv, y_cv = X_all[:80], y_all[:80]
# grid over alpha and l1_ratio for ElasticNet
a_grid, l_grid = [0.01, 0.03, 0.1, 0.3, 1.0], [0.1, 0.5, 0.9, 1.0]
G = np.array([[kfold_cv(lambda: ElasticNet(a, l, n_iters=60), X_cv, y_cv) for a in a_grid] for l in l_grid])
i, j = np.unravel_index(G.argmin(), G.shape)
print(f"CV grid best: l1_ratio={l_grid[i]}, alpha={a_grid[j]}, CV MSE={G.min():.2f}")
plt.figure(figsize=(5.5, 4)); plt.imshow(G, cmap='viridis'); plt.colorbar(label='CV MSE')
plt.xticks(range(5), a_grid); plt.yticks(range(4), l_grid); plt.xlabel('alpha'); plt.ylabel('l1_ratio')
plt.title('ElasticNet CV grid'); plt.tight_layout(); plt.savefig('plots/04_cv_grid.png', dpi=120); plt.close()

## 8. Final comparison on the test set, and correlated features (Lasso picks one, ElasticNet groups)

In [ ]:
best = {nm: al[np.argmin(v)] for nm, (t, v) in curves.items()}
models = {'OLS': Ridge(0.0), 'Ridge': Ridge(best['Ridge']),
          'Lasso': ElasticNet(best['Lasso'], 1.0, n_iters=300), 'ElasticNet': ElasticNet(best['ElasticNet'], 0.5, n_iters=300)}
print(f"{'model':11s} {'test MSE':>9s} {'test R2':>8s} {'nonzero':>8s}")
res = {}
for nm, m_ in models.items():
    m_.fit(Xs_tr, y_tr); p = m_.predict(Xs_te); res[nm] = mse(y_te, p)
    print(f"{nm:11s} {mse(y_te, p):9.2f} {r2(y_te, p):8.3f} {nnz(m_.w):8d}")

# a group of 3 near-identical informative features
n = 200; z = rng.normal(size=n)
Xg = np.c_[z + 0.05 * rng.normal(size=n), z + 0.05 * rng.normal(size=n), z + 0.05 * rng.normal(size=n), rng.normal(size=(n, 3))]
yg = 3 * z + rng.normal(0, 1, n); Xg, _, _ = standardize(Xg)
wl = ElasticNet(0.3, 1.0).fit(Xg, yg).w; we = ElasticNet(0.3, 0.3).fit(Xg, yg).w
print("Correlated group  Lasso:", wl.round(2), "\n                  ElasticNet:", we.round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(res.keys(), res.values()); ax[0].set_ylabel('test MSE'); ax[0].set_title('Test error')
xw = np.arange(20)
for off, (nm, m_) in zip([-.3, -.1, .1, .3], models.items()): ax[1].bar(xw + off, m_.w, width=.2, label=nm)
ax[1].bar(xw, w_true, width=.05, color='k', label='true'); ax[1].legend(fontsize=7); ax[1].set_xlabel('feature'); ax[1].set_title('Learned weights')
plt.tight_layout(); plt.savefig('plots/05_final_comparison.png', dpi=120); plt.close()

## 9. Optional sanity check: our Ridge vs numpy lstsq on augmented system

In [ ]:
m = len(y_tr); a = 0.1
Aug = np.r_[Xs_tr, np.sqrt(m * a) * np.eye(20)]; bb = np.r_[y_tr - y_tr.mean(), np.zeros(20)]
w_ls = np.linalg.lstsq(Aug, bb, rcond=None)[0]
print("Ridge vs lstsq(augmented) diff:", np.abs(w_ls - r_cf.w).max().round(8))